<a href="https://colab.research.google.com/github/ldongheedev/-BDA-LLM-RAG-Program/blob/main/%EC%9D%B4%EB%8F%99%ED%9D%AC_7268_9%EC%A3%BC%EC%B0%A8_%EB%B3%B5%EC%8A%B5%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 9주차 복습 과제: Gemini API와 LangChain

> **복습 키워드**
> 1. Gemini API와 기본 API를 불러오는 구조 이해하기
> 2. API를 활용하여 언어모델을 이용하는 것에 익숙해지기


---
## 0. 환경 설정

Gemini API를 사용하기 위해 필요한 라이브러리를 설치합니다.

- `google-genai`: Google Gemini 공식 SDK (2025~ 신규 버전)
- `langchain`: LLM 애플리케이션 개발 프레임워크
- `langchain-google-genai`: LangChain에서 Gemini를 사용하기 위한 연결 라이브러리


In [ ]:
!pip install -q google-genai langchain langchain-google-genai


API Key는 [Google AI Studio](https://aistudio.google.com)에서 무료로 발급받을 수 있습니다.
Google 계정만 있으면 되고, 신용카드는 필요 없습니다.

발급 경로: AI Studio 접속 → "Get API key" → "Create API key"


In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "api 키 여기다 작성하기"

# 주의: API 키를 GitHub 등에 올리지 마세요!


---
## 1. Gemini API 기본 호출 구조

Gemini API의 호출 구조는 3단계입니다.

```
1. Client 생성 (API 키로 인증)
2. generate_content() 호출 (모델명 + 질문)
3. response.text로 응답 확인
```

이것이 LLM API를 사용하는 가장 기본적인 패턴입니다.


In [ ]:
# 가장 기본적인 호출
from google import genai

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

response = client.models.generate_content(
    model="gemini-2.0-flash",                          # 사용할 모델
    contents="텍스트 분석이 뭔지 한 줄로 설명해줘",     # 질문
)

print(f"응답: {response.text}")


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 47.705441298s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.go

### config 옵션으로 모델 동작 제어하기

`GenerateContentConfig`를 사용하면 모델의 동작을 세밀하게 조절할 수 있습니다.

| 옵션 | 역할 | 값 |
|------|------|----|
| `system_instruction` | 모델에게 역할/규칙을 부여 | 문자열 |
| `temperature` | 창의성 조절 (낮을수록 보수적) | 0 ~ 2 |
| `max_output_tokens` | 최대 생성 길이 제한 | 정수 |


In [ ]:
# system_instruction + temperature 설정
from google.genai import types

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="리뷰: 이 영화 정말 재미있다",
    config=types.GenerateContentConfig(
        system_instruction="영화 리뷰의 감성을 긍정/부정/중립 중 하나만 답하세요.",
        temperature=0,             # 분류 작업은 항상 같은 답이 나오도록 0
        max_output_tokens=100,
    ),
)

print(f"감성: {response.text}")


### 반복문으로 여러 리뷰 분류하기

같은 API 호출 구조를 반복문에 넣으면 여러 텍스트를 한 번에 처리할 수 있습니다.


In [ ]:
# 여러 리뷰를 반복문으로 분류
reviews = [
    "배우 연기가 훌륭하고 스토리도 좋다",
    "시간 낭비 다시는 안 봄",
    "그저 그랬다 볼만은 했음",
    "배우 연기는 좋은데 스토리가 아쉽다",
]

print("=== Gemini 감성 분류 ===")
for review in reviews:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"리뷰: {review}",
        config=types.GenerateContentConfig(
            system_instruction="리뷰의 감성을 긍정/부정/중립 중 하나만 답하세요. 한 단어만 출력.",
            temperature=0,
        ),
    )
    print(f"  {review} → {response.text.strip()}")


---
## 2. API 호출을 함수로 만들어 재사용하기

같은 호출 코드를 반복하는 대신 **함수**로 만들면 코드가 깔끔해지고 재사용이 쉬워집니다.
이 패턴은 실제 LLM 애플리케이션 개발에서 가장 기본이 되는 구조입니다.


In [ ]:
import time

# 감성 분류 함수
def classify_sentiment(review):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"리뷰: {review}",
        config=types.GenerateContentConfig(
            system_instruction="리뷰의 감성을 긍정/부정/중립 중 하나만 답하세요.",
            temperature=0,
        ),
    )
    return response.text.strip()


test_reviews = [
    "가격 대비 만족합니다",
    "포장이 엉망이고 제품이 파손됐어요",
    "디자인은 좋은데 내구성이 약해요",
]

for review in test_reviews:
    result = classify_sentiment(review)
    print(f"  '{review}' → {result}")
    time.sleep(15)


### JSON 구조화 출력

system_instruction에서 JSON 형식을 지정하면, 감성/키워드/요약을 한 번의 호출로 동시에 추출할 수 있습니다.
응답을 `json.loads()`로 파싱하면 Python dict로 바로 사용 가능합니다.


In [ ]:
# JSON 구조화 출력 함수
import json

def analyze_review(review):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"리뷰: {review}",
        config=types.GenerateContentConfig(
            system_instruction=(
                "리뷰를 분석하여 아래 JSON 형식으로만 응답하세요. "
                "다른 텍스트 없이 순수 JSON만 출력하세요.\n"
                '{"sentiment": "긍정/부정/중립", "keywords": ["키워드1", "키워드2"], "summary": "한 줄 요약"}'
            ),
            temperature=0,
        ),
    )
    text = response.text.strip()
    text = text.replace("```json", "").replace("```", "").strip()
    return json.loads(text)


result = analyze_review("배우 연기는 훌륭한데 스토리가 진부하고 결말이 허무했다")

print("=== 리뷰 분석 결과 (JSON) ===")
print(json.dumps(result, ensure_ascii=False, indent=2))
print(f"\n감성: {result['sentiment']}")
print(f"키워드: {result['keywords']}")


In [ ]:
# 여러 리뷰를 JSON으로 분석
test_reviews = [
    "배송 빠르고 포장 꼼꼼해요 재구매 의사 있습니다",
    "색상이 사진과 달라서 실망했어요",
    "가격은 괜찮은데 소재가 좀 아쉽네요",
]

print("=== 여러 리뷰 JSON 분석 ===")
for review in test_reviews:
    result = analyze_review(review)
    print(f"\n리뷰: {review}")
    print(f"  감성: {result['sentiment']}")
    print(f"  키워드: {result['keywords']}")
    print(f"  요약: {result['summary']}")


---
## 3. Temperature에 따른 응답 차이

temperature는 모델의 "창의성"을 조절하는 파라미터입니다.

| 값 | 특징 | 적합한 용도 |
|-----|------|------------|
| 0 | 항상 같은 결과 (결정적) | 분류, 정보 추출 |
| 0.5 | 적당한 다양성 | 일반 대화 |
| 1.0 | 다양한 표현 | 글쓰기, 창작 |
| 1.5+ | 매우 창의적, 예측 불가 | 브레인스토밍 |


In [ ]:
# 같은 질문에 temperature를 바꿔가며 비교
question = "인공지능의 미래에 대해 한 문장으로 말해주세요."

for temp in [0, 0.5, 1.0, 1.5]:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=question,
        config=types.GenerateContentConfig(
            temperature=temp,
            max_output_tokens=100,
        ),
    )
    print(f"temp={temp}: {response.text.strip()[:100]}")
    print()


---
## 4. LangChain LCEL 기초

LangChain은 LLM 애플리케이션을 쉽게 만들 수 있는 프레임워크입니다.
핵심 문법은 **LCEL (LangChain Expression Language)**로, 파이프(`|`)로 단계를 연결합니다.

```
chain = prompt | llm | parser
```

| 구성 요소 | 역할 | 설명 |
|-----------|------|------|
| `prompt` | 입력 템플릿 | 변수를 채워서 메시지를 생성 |
| `llm` | 언어 모델 | 메시지를 Gemini에 보내고 응답을 받음 |
| `parser` | 출력 파서 | 응답에서 필요한 부분만 추출 |

Gemini API를 직접 호출하는 것과 같은 결과를 얻지만,
LangChain을 사용하면 **변수 관리, 체인 연결, 병렬 처리**가 훨씬 편리해집니다.


In [ ]:
# LangChain 기본 체인: prompt | llm | parser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# (1) 프롬프트 템플릿 — {review}가 변수
prompt = ChatPromptTemplate.from_messages([
    ("system", "리뷰의 감성을 긍정/부정/중립 중 하나만 답하세요."),
    ("user", "리뷰: {review}"),
])

# (2) LLM — Gemini 모델
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# (3) 출력 파서 — 문자열 그대로 반환
parser = StrOutputParser()

# (4) 체인 연결
chain = prompt | llm | parser

# 실행
result = chain.invoke({"review": "이 영화 정말 재미있다"})
print(f"결과: {result}")


### chain.invoke()가 내부적으로 하는 일

`chain.invoke()`는 3단계를 순서대로 실행합니다.
아래 코드에서 각 단계를 분리해서 확인할 수 있습니다.


In [ ]:
# 각 단계별 동작 확인

# [Step 1] prompt: {review} 변수를 채워서 메시지 생성
messages = prompt.invoke({"review": "이 영화 정말 재미있다"})
print("[Step 1] prompt.invoke() →")
for msg in messages.messages:
    print(f"  {msg.type}: {msg.content}")

# [Step 2] llm: 메시지를 Gemini에 보내고 응답
llm_response = llm.invoke(messages)
print(f"\n[Step 2] llm.invoke() →")
print(f"  타입: {type(llm_response).__name__}")
print(f"  내용: {llm_response.content}")

# [Step 3] parser: AIMessage에서 문자열만 추출
parsed = parser.invoke(llm_response)
print(f"\n[Step 3] parser.invoke() →")
print(f"  {parsed}")

print(f"\n→ chain.invoke()는 이 3단계를 한 번에 실행합니다")


### 프롬프트에 여러 변수 사용하기

system 메시지에도 `{변수}`를 넣을 수 있습니다.
같은 질문이라도 역할을 바꾸면 다른 관점의 답변을 얻을 수 있습니다.


In [ ]:
# 여러 변수 사용 + 역할 바꾸기
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role}입니다. 한국어로 간결하게 답변하세요."),
    ("user", "{question}"),
])

chain = prompt | llm | StrOutputParser()

question = "인공지능의 미래는?"

for role in ["데이터 사이언티스트", "초등학교 선생님", "SF 소설가"]:
    result = chain.invoke({"role": role, "question": question})
    print(f"[{role}]")
    print(f"  {result[:100]}...")
    print()


### batch — 여러 입력을 한 번에 병렬 처리

`chain.invoke()`는 하나의 입력을 처리하지만,
`chain.batch()`는 여러 입력을 리스트로 받아 병렬로 처리합니다.
하나씩 호출하는 것보다 훨씬 빠릅니다.


In [ ]:
# batch로 여러 리뷰 한 번에 처리
prompt = ChatPromptTemplate.from_messages([
    ("system", "리뷰의 감성을 긍정/부정/중립 중 하나만 답하세요."),
    ("user", "리뷰: {review}"),
])
chain = prompt | llm | StrOutputParser()

reviews = [
    {"review": "이 영화 정말 재미있다"},
    {"review": "시간 낭비 최악이다"},
    {"review": "그저 그랬다"},
    {"review": "배우 연기가 훌륭하다"},
    {"review": "지루하고 별로다"},
]

results = chain.batch(reviews)

print("=== batch 결과 ===")
for r, result in zip(reviews, results):
    print(f"  {r['review']:20s} → {result}")


---
## 5. LangChain 심화

### JsonOutputParser — JSON 응답을 dict로 자동 변환

`StrOutputParser`는 문자열을 그대로 반환하지만,
`JsonOutputParser`는 JSON 문자열을 Python dict로 자동 변환해줍니다.
감성 + 키워드 + 요약 등 여러 정보를 한 번에 구조화하여 추출할 때 유용합니다.


In [ ]:
# JsonOutputParser로 JSON → dict 자동 변환
from langchain_core.output_parsers import JsonOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", """리뷰를 분석하여 아래 JSON으로만 응답하세요. 다른 텍스트 없이 JSON만.
{{"sentiment": "긍정/부정/중립", "keywords": ["키워드1", "키워드2"], "summary": "한줄요약"}}"""),
    ("user", "리뷰: {review}"),
])

chain = prompt | llm | JsonOutputParser()

result = chain.invoke({"review": "배우 연기는 좋은데 스토리가 너무 진부하다"})

print(f"타입: {type(result)}")    # dict!
print(f"감성:   {result['sentiment']}")
print(f"키워드: {result['keywords']}")
print(f"요약:   {result['summary']}")


### 다단계 파이프라인

하나의 체인 출력을 다른 체인의 입력으로 연결하면 **다단계 처리**가 가능합니다.
아래 예시는 "긴 텍스트 → 한국어 요약 → 영어 번역" 2단계 파이프라인입니다.


In [ ]:
# 다단계 파이프라인: 요약 → 번역

# 1단계: 요약 체인
prompt_summary = ChatPromptTemplate.from_messages([
    ("system", "주어진 텍스트를 2줄로 요약하세요."),
    ("user", "{text}"),
])
chain_summary = prompt_summary | llm | StrOutputParser()

# 2단계: 번역 체인
prompt_translate = ChatPromptTemplate.from_messages([
    ("system", "주어진 한국어 텍스트를 영어로 번역하세요."),
    ("user", "{text}"),
])
chain_translate = prompt_translate | llm | StrOutputParser()

# 실행
long_text = """인공지능 기술이 빠르게 발전하면서 다양한 산업에 혁신을 가져오고 있다.
특히 자연어 처리 분야에서는 대규모 언어 모델이 등장하면서
번역, 요약, 대화 시스템 등에서 놀라운 성능을 보이고 있다.
하지만 할루시네이션, 편향성 등의 문제도 해결해야 할 과제로 남아있다."""

# 1단계 실행
summary = chain_summary.invoke({"text": long_text})
print(f"[1단계] 요약:\n  {summary}")

# 2단계 실행 (1단계 결과를 입력으로 사용)
translation = chain_translate.invoke({"text": summary})
print(f"\n[2단계] 영어 번역:\n  {translation}")


---
## 6. 실습: Gemini API로 뉴스 카테고리 분류

지금까지 배운 Gemini API 호출 구조를 활용하여 뉴스 기사를 카테고리별로 분류합니다.
system_instruction으로 분류 기준을 설정하고, temperature=0으로 일관된 결과를 얻습니다.


In [ ]:
# 뉴스 카테고리 분류
articles = [
    "삼성전자가 새로운 AI 반도체 칩을 발표했다. 성능이 기존 대비 2배 향상됐다.",
    "정부가 내년도 예산안을 국회에 제출했다. 복지 분야 지출이 크게 늘었다.",
    "BTS가 신곡을 발표하며 전 세계 차트 1위를 기록했다.",
]

print("=== 뉴스 카테고리 분류 ===")
for article in articles:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"기사: {article}",
        config=types.GenerateContentConfig(
            system_instruction="뉴스 기사를 정치/경제/사회/문화/IT 중 하나로 분류하세요. 카테고리만 답하세요.",
            temperature=0,
        ),
    )
    print(f"  [{response.text.strip()}] {article[:40]}...")


## 7. 실습: LangChain 리뷰 분석 파이프라인

LangChain의 LCEL + JsonOutputParser + batch를 결합하여
여러 리뷰를 한 번에 분석하는 파이프라인을 구성합니다.


In [ ]:
# LangChain + JsonOutputParser + batch
from langchain_core.output_parsers import JsonOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", """리뷰를 분석하여 아래 JSON으로만 응답하세요.
{{"sentiment": "긍정/부정/중립", "score": 1~5, "keywords": ["키워드1", "키워드2"]}}"""),
    ("user", "리뷰: {review}"),
])

chain = prompt | llm | JsonOutputParser()

test_reviews = [
    {"review": "배송 빠르고 품질도 좋아요"},
    {"review": "제품이 불량이고 환불도 안 돼요"},
    {"review": "가격은 괜찮은데 색상이 사진과 달라요"},
    {"review": "이 가격에 이 품질이면 최고입니다"},
    {"review": "한 달 만에 고장났어요"},
]

results = chain.batch(test_reviews)

print("=== 리뷰 분석 결과 ===")
for review, result in zip(test_reviews, results):
    print(f"\n  리뷰: {review['review']}")
    print(f"  감성: {result['sentiment']} | 점수: {result['score']} | 키워드: {result['keywords']}")


## 8. 실습: 다단계 파이프라인 (키워드 추출 → 개선 제안)

1단계에서 리뷰의 핵심 키워드를 추출하고,
2단계에서 그 키워드를 기반으로 서비스 개선 제안을 생성합니다.
이처럼 체인을 연결하면 복잡한 분석 작업도 단계별로 처리할 수 있습니다.


In [ ]:
# 1단계: 키워드 추출 → 2단계: 개선 제안

# 1단계 체인
prompt_kw = ChatPromptTemplate.from_messages([
    ("system", "리뷰에서 핵심 키워드 3개를 추출하세요. 쉼표로 구분, 키워드만 답하세요."),
    ("user", "리뷰: {review}"),
])
chain_kw = prompt_kw | llm | StrOutputParser()

# 2단계 체인
prompt_suggest = ChatPromptTemplate.from_messages([
    ("system", "아래 키워드를 바탕으로 서비스 개선 제안을 한 줄로 작성하세요."),
    ("user", "키워드: {keywords}"),
])
chain_suggest = prompt_suggest | llm | StrOutputParser()

# 여러 리뷰에 대해 실행
reviews = [
    "배송은 빠른데 포장이 엉망이고 제품에 스크래치가 있었다",
    "앱이 자꾸 튕기고 로딩이 너무 느려요",
    "상담원 연결이 안 되고 자동 응답만 반복돼요",
]

print("=== 키워드 추출 → 개선 제안 ===")
for review in reviews:
    keywords = chain_kw.invoke({"review": review})
    suggestion = chain_suggest.invoke({"keywords": keywords})
    print(f"\n리뷰: {review}")
    print(f"  키워드: {keywords}")
    print(f"  개선 제안: {suggestion}")


---
## 핵심 정리

### Gemini API 호출 구조

```python
from google import genai
from google.genai import types

client = genai.Client(api_key="KEY")
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="질문",
    config=types.GenerateContentConfig(
        system_instruction="역할 설정",
        temperature=0,
        max_output_tokens=100,
    ),
)
print(response.text)
```

### LangChain LCEL 구조

```python
chain = prompt | llm | parser         # 체인 정의
result = chain.invoke({"변수": "값"})  # 단일 실행
results = chain.batch([...])           # 병렬 실행
```

### Gemini API vs LangChain 비교

| 구분 | Gemini API 직접 호출 | LangChain |
|------|---------------------|----------|
| 장점 | 코드가 단순, 이해하기 쉬움 | 변수 관리, 체인 연결, batch 편리 |
| 적합 | 단순한 단일 호출 | 복잡한 파이프라인, 여러 입력 처리 |

### 주요 파서

| 파서 | 용도 |
|------|------|
| `StrOutputParser()` | 문자열 그대로 반환 |
| `JsonOutputParser()` | JSON → Python dict 자동 변환 |
